In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression,SGDRegressor
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import  CountVectorizer
from sklearn.ensemble import (RandomForestClassifier,RandomForestRegressor,HistGradientBoostingClassifier)
from sklearn.naive_bayes import (BernoulliNB,MultinomialNB,GaussianNB)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier




# Loading E-commerce Dataset

## Dropping Unimportant Columns & rows


In [2]:
Ecom_data=pd.read_csv("Pakistan Largest Ecommerce Dataset.csv")
Ecom_data.drop(Ecom_data.iloc[:,[0,3,7,9,18,19,21,22,23,24,25,20,17,13,12]],axis=1,inplace=True)


C:\Users\Hamza Sajid\AppData\Local\Temp\ipykernel_13536\170460723.py:1: DtypeWarning: Columns (0: status, 1: created_at, 2: sku, 3: increment_id, 4: category_name_1, 5: sales_commission_code, 6: payment_method, 7: Working Date, 8: BI Status, 9:  MV , 10: Customer Since, 11: M-Y, 12: FY) have mixed types. Specify dtype option on import or set low_memory=False.
  Ecom_data=pd.read_csv("Pakistan Largest Ecommerce Dataset.csv")


In [3]:
Ecom_data['created_at']=pd.to_datetime(Ecom_data['created_at'])
Ecom_data['Order_day']=Ecom_data['created_at'].dt.day
Ecom_data.drop('created_at',axis=1,inplace=True)


## Handling - Nan Values

In [4]:
def rename(new,old):
    return Ecom_data.rename(columns={new:old},inplace=True)
rename('Year','Order_year')
rename('Month','Order_month')
Ecom_data.dropna(how='all',inplace=True)

In [5]:
Ecom_data['status']=Ecom_data['status'].fillna('complete')
Ecom_data['category_name_1']=Ecom_data['category_name_1'].fillna('Mobiles & Tablets')

## Feature Engineering (grouping Status)

In [6]:
Completed_order=['complete','closed','paid',]
Cancelled_order=['refund','canceled','order_refunded','exchange']
Inprocess_order=['cod','pending_paypal','processing','payment_review','holded','received','pending']
Risk_order=['fraud']
def filter(x):
    if x in Completed_order:
        return 'Completed'
    elif x in Cancelled_order:
        return 'Cancelled'
    elif x in Inprocess_order:
        return 'Inprocess'
    elif x in Risk_order:
        return 'Fraud'
    else:
        return 'Unknown'
values=Ecom_data['status'].apply(filter)

In [7]:
Ecom_data['status']=values
Ecom_data

,status,price,qty_ordered,grand_total,category_name_1,discount_amount,payment_method,MV,Order_year,Order_month,Order_day
0,Completed,1950.0,1.0,1950.0,Women's Fashion,0.0,cod,"1,950",2016.0,7.0,1.0
1,Cancelled,240.0,1.0,240.0,Beauty & Grooming,0.0,cod,240,2016.0,7.0,1.0
2,Cancelled,2450.0,1.0,2450.0,Women's Fashion,0.0,cod,"2,450",2016.0,7.0,1.0
3,Completed,360.0,1.0,60.0,Beauty & Grooming,300.0,cod,360,2016.0,7.0,1.0
4,Cancelled,555.0,2.0,1110.0,Soghaat,0.0,cod,"1,110",2016.0,7.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
584519,Inprocess,699.0,1.0,849.0,Women's Fashion,0.0,cod,699,2018.0,8.0,28.0
584520,Inprocess,35599.0,1.0,35899.0,Mobiles & Tablets,0.0,bankalfalah,"35,599",2018.0,8.0,28.0
584521,Inprocess,129999.0,2.0,652178.0,Mobiles & Tablets,0.0,bankalfalah,"259,998",2018.0,8.0,28.0
584522,Inprocess,87300.0,2.0,652178.0,Mobiles & Tablets,0.0,bankalfalah,"174,600",2018.0,8.0,28.0


## grouping Similar product Categories

In [8]:
Electronics=['Mobiles & Tablets','Computing','Appliances']
Fashion=["Men's Fashion","Women's Fashion",'Beauty & Grooming','Home & Living']
essentials=['Kids & Baby','Superstore','Health & Sports','Soghaat','Others']
Learn=['Entertainment','School & Education','Books']
def category(x):
    if x in Electronics:
        return 'Electronics & Tech'
    elif x in Fashion:
        return 'Fashion & Lifestyle'
    elif x in essentials:
        return 'Family & Essentials'
    elif x in Learn:
        return 'Learning & Entertainment'
    else:
        return 'Other'
new_cat=Ecom_data['category_name_1'].apply(category)

In [9]:
Ecom_data['category_name_1']=new_cat
Ecom_data

,status,price,qty_ordered,grand_total,category_name_1,discount_amount,payment_method,MV,Order_year,Order_month,Order_day
0,Completed,1950.0,1.0,1950.0,Fashion & Lifestyle,0.0,cod,"1,950",2016.0,7.0,1.0
1,Cancelled,240.0,1.0,240.0,Fashion & Lifestyle,0.0,cod,240,2016.0,7.0,1.0
2,Cancelled,2450.0,1.0,2450.0,Fashion & Lifestyle,0.0,cod,"2,450",2016.0,7.0,1.0
3,Completed,360.0,1.0,60.0,Fashion & Lifestyle,300.0,cod,360,2016.0,7.0,1.0
4,Cancelled,555.0,2.0,1110.0,Family & Essentials,0.0,cod,"1,110",2016.0,7.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
584519,Inprocess,699.0,1.0,849.0,Fashion & Lifestyle,0.0,cod,699,2018.0,8.0,28.0
584520,Inprocess,35599.0,1.0,35899.0,Electronics & Tech,0.0,bankalfalah,"35,599",2018.0,8.0,28.0
584521,Inprocess,129999.0,2.0,652178.0,Electronics & Tech,0.0,bankalfalah,"259,998",2018.0,8.0,28.0
584522,Inprocess,87300.0,2.0,652178.0,Electronics & Tech,0.0,bankalfalah,"174,600",2018.0,8.0,28.0


## Grouping Similar Transaction Processes 

In [10]:
Cod=['cod','cashatdoorstep']
Online=['Payaxis','ublcreditcard','mygateway','mcblite','internetbanking','jazzvoucher','jazzwallet','Easypay','Easypay_MA','easypay_voucher', 'bankalfalah','apg']
credit=['customercredit','productcredit']
def transaction(x):
    if x in Cod:
        return 'COD'
    elif x in Online:
        return 'Online Transfer'
    elif x in credit:
        return 'Pay later'
    else:
        return 'Other'
pay=Ecom_data['payment_method'].apply(transaction)

## Columns Corrections (dtype, Correction of Grand total Prices)

In [11]:
Ecom_data['payment_method']=pay
Ecom_data[['Order_year','Order_month','Order_day','qty_ordered']]=Ecom_data[['Order_year','Order_month','Order_day','qty_ordered']].astype('int64')
Ecom_data['grand_total']=(Ecom_data['price']*Ecom_data['qty_ordered'])-abs(Ecom_data['discount_amount'])
Indexes=Ecom_data[Ecom_data['payment_method']=='Pay later'].index
Ecom_data.loc[Indexes,'grand_total']=0.0
Ecom_data[' MV ']=Ecom_data['price']*Ecom_data['qty_ordered']

## FIlling Prices on the basis of Products Category Averages

In [12]:
Mean_prices=Ecom_data.groupby('category_name_1')['price'].mean()
for i in Mean_prices.index:
    print(i)
    Price_index=Ecom_data[(Ecom_data['price']==0) & (Ecom_data['category_name_1']==i)].index
    Ecom_data.loc[Price_index,'price']=Mean_prices[i]

Electronics & Tech
Family & Essentials
Fashion & Lifestyle
Learning & Entertainment
Other


## Again calculation with new values

In [13]:
Ecom_data[(Ecom_data['price']==Mean_prices['Electronics & Tech']) & (Ecom_data['category_name_1']=='Electronics & Tech')]
Ecom_data['grand_total']=(Ecom_data['price']*Ecom_data['qty_ordered'])-abs(Ecom_data['discount_amount'])
Indexes=Ecom_data[Ecom_data['payment_method']=='Pay later'].index
Ecom_data.loc[Indexes,'grand_total']=0.0
Ecom_data[' MV ']=Ecom_data['price']*Ecom_data['qty_ordered']

## Detecting Outliers

In [14]:
def outlier(x):
    q1,q3=Ecom_data[x].quantile([0.25,0.75])
    Iqr=q3-q1
    lower=q1-1.5*Iqr
    Upper=q3+1.5*Iqr
    return lower,Upper
Drop_price_index=Ecom_data[Ecom_data.price<1].index
Ecom_data.drop(Drop_price_index,axis=0,inplace=True)
Ecom_data.loc[[152474,152478,152479],'discount_amount']=[599.5,2.0,2.0]

In [17]:
Ecom_data.to_csv('Cleaned E-commerce Dataset.csv',index=False)
Mean_prices

category_name_1
Electronics & Tech          15107.834074
Family & Essentials           927.261657
Fashion & Lifestyle          1160.130364
Learning & Entertainment    16086.201274
Other                        4147.859827
Name: price, dtype: float64